# Desocupação nas unidades da Federação — 2024 e 2025

Fonte: IBGE, PNAD Contínua. Recorte intencional de 12 UFs presentes nos dois textos do IBGE; não representa todas as UFs. Taxas anuais (%). Execute no Colab a partir do arquivo CSV incluído.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

caminho = Path("../dados/desocupacao_ibge_2024_2025.csv")
if not caminho.exists():
    from google.colab import files
    print("Selecione o arquivo desocupacao_ibge_2024_2025.csv")
    files.upload()
    caminho = Path("desocupacao_ibge_2024_2025.csv")
df = pd.read_csv(caminho)
df.head()

## Limpeza e validação

In [ ]:
print("Linhas e colunas:", df.shape)
print("Tipos:\n", df.dtypes)
print("Ausentes:\n", df.isna().sum())
print("Duplicatas exatas:", df.duplicated().sum())
print("Duplicatas UF e ano:", df.duplicated(["uf", "ano"]).sum())
df["uf"] = df["uf"].str.strip()
df["regiao"] = df["regiao"].str.strip()
df["taxa_desocupacao_pct"] = pd.to_numeric(df["taxa_desocupacao_pct"], errors="coerce")
assert df["taxa_desocupacao_pct"].between(0, 100).all()
assert not df.duplicated(["uf", "ano"]).any()
assert df.groupby("uf")["ano"].nunique().eq(2).all()

## Distribuição e evolução

In [ ]:
display(df.groupby("ano")["taxa_desocupacao_pct"].agg(["count", "mean", "median", "min", "max"]).round(2))
tabela = df.pivot(index="uf", columns="ano", values="taxa_desocupacao_pct")
tabela["variacao_pp"] = tabela[2025] - tabela[2024]
display(tabela.sort_values("variacao_pp").round(2))
print("Correlação entre as taxas por UF nos dois anos:", round(tabela[2024].corr(tabela[2025]), 3))
print("Quantidade de UFs com redução:", (tabela.variacao_pp < 0).sum())

In [ ]:
tabela[[2024, 2025]].sort_values(2025).plot.barh(figsize=(9,6), color=["#9bb7d4", "#205a8e"])
plt.title("Taxa anual de desocupação: 12 UFs selecionadas")
plt.xlabel("Taxa (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

## Interpretação
Compare as diferenças em pontos percentuais. Os resultados são descritivos: não demonstram causas, e as médias simples entre UFs não correspondem à taxa do Brasil. Não extrapole o recorte para as 27 UFs.